<center><img src="https://i.pinimg.com/webp/1200x/47/6f/07/476f07bee96bdbc9b0c97acbc01175d3.webp" width="1200" height="300"></center>

## <b>1 <span style='color:#F1A424'>|</span> Nice One Beauty Market Performance (2025–2026)
</b> 

<div style="color:white;display:fill;border-radius:8px;font-size:100%; letter-spacing:1.0px;"><p style="padding: 5px;color:white;text-align:left;"><b><span style='color:#F1A424'>WHAT WE WILL DO IN THIS SECTION</span></b></p></div>

- 시작일 : 2026-06-10 수요일 오후 6시 
- 데이터셋 : Nice One Beauty (4193.SR) Tadawul Stock Price E-Commerce Market Dataset (2025-2026)
- 키워드 :


<div style="color:white;display:fill;border-radius:8px;background-color:#323232;font-size:150%; letter-spacing:1.0px"><p style="padding: 12px;color:white;"><b><b><span style='color:white'><span style='color:#F1A424'>1.1 | </span></span></b> 프로젝트 개요 및 도메인</b></p></div>

**잘나가던 중동의 뷰티 스타트업이 상장 후 왜 주가가 폭등했다가 반토막 이하로 폭락했는가?**

- 회사명 : 나이스원 뷰티(Nice One Beauty)
    - 중동 지역의 이커머스 화장품 마케팅 브랜드
    - 인플루언서 마켓으로 시작 자체적인 모바일 app, 웹스토어 개발
    - 현재는 주문의 95% 온라인

- 배경 
    - 2025년 1월에 상장 35SAR로 지분 30프로 ➔ 상장 후 6주만에 주가가 68.4 까지 95프로 급등
    - 경영의 현실 (Operational Reality): 지역 유통 허브(제다 메가 물류창고 등) 구축을 위한 자본 지출(CapEx), 고객 경험 쇼룸 운영, 그리고 높은 마케팅 고객 획득 비용(CAC)으로 인해 영업 비용이 누적되면서 수익성이 압박
    - 시장 조정 (Market Correction) ➔  2026년 1월 초 : 16.8 떡락. 연말에는 17.4 로 안정화 

<div style="color:white;display:fill;border-radius:8px;font-size:100%; letter-spacing:1.0px;"><p style="padding: 5px;color:white;text-align:left;"><b><span style='color:#F1A424'>다루게 될 데이터와 분석 방향.</span></b></p></div>

1. **주가 데이터 분석 (시계열)** : 공모가(35 SAR) $\rightarrow$ 최고가(68.40 SAR) $\rightarrow$ 최저가(16.80 SAR)로 이어지는 주가 변동 추이 시각화 및 기술 통계
2. **재무 및 비용 데이터 분석** : 매출은 나는데 왜 주가가 떨어졌는지 |증명하기 위해, 물류창고 짓느라 쓴 돈(CapEx)과 마케팅 비용(CAC)이 마진(수익률)에 미친 영향 분석.
3. **시장 심리 및 지지선 분석** : 주식 시장에서 '주요 지지선'이 깨진 순간을 통계적 분석이나 주식 보조지표(이동평균선 등)로 시각화.


<div style="color:white;display:fill;border-radius:8px;background-color:#323232;font-size:150%; letter-spacing:1.0px"><p style="padding: 12px;color:white;"><b><b><span style='color:white'><span style='color:#F1A424'>1.2 | 
</span></span></b> 환경 설정 및 라이브러리 임포트</b></p></div>


In [26]:
!uv pip install pandas numpy matplotlib seaborn plotly statsmodels

Checked 6 packages in 983ms


In [ ]:
# 1.2.1 : 라이브러리 임포트

# 데이터 분석용 라이브러리
import pandas as pd
import numpy as np

# EDA 및 데이터시각화용 라이브러리
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 통계검증용 라이브러리
from statsmodels.tsa.stattools import adfuller

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 1.2.2 : 시각화 스타일 설정

# 그래프의 배경을 흰색/회식 격자무늬로 세팅
sns.set_theme(style="whitegrid")
# 그래프 사이즈의 기본 크기를 가로 12, 세로 6인치 고정
plt.rcParams['figure.figsize'] = [12,6]
# 해상도(DPI)를 100으로 설정해서 선명하게 출력되도록 고정
plt.rcParams['figure.dpi'] = 100
# 그래프에 들어갈 폰트 지정
plt.rcParams['font.family'] = 'Malgun Gothic'

print("모든 라이브러리 임포트 완료입니다")

모든 라이브러리 임포트 완료입니다


<div style="color:white;display:fill;border-radius:8px;background-color:#323232;font-size:150%; letter-spacing:1.0px"><p style="padding: 12px;color:white;"><b><b><span style='color:white'><span style='color:#F1A424'>1.3 | </span></span></b> 데이터 로드 & 전처리</b></p></div>

- 원본 데이터셋(csv) 로드
- 전처리 단계:
    1. Date(날짜) 컬럼을 datetime(날짜형) 형식으로 변환 
    2. 시간 순서대로 정렬 (옛날 날짜부터 가장 최신 날짜까지)
    3. 파생 변수 생성 
        1. '일일 수익률(Daily Return)' 및 '로그 수익률(Log Return)'
        2. 변동성 지표: 일일 수익률의 7일 rolling 표준편차
        3. 변동성 지표: 일일 수익률의 30일 표준편차 <br> rolling window = 7 / 30
        4. 시간 관련 메타데이터 추출 (연, 월, 월 이름, 요일, 분기) <br> .dt.
    4. 정상성 확인 (Sanity Check): df.shape, 결측치 검사, df.head() 확인

**<mark style="background-color: #ffdf3d;color:black;border-radius:5px;opacity:1.0">1. 데이터셋 로드 및 컬럼 별 dtype 확인 </mark>** 

In [29]:
df = pd.read_csv("data/Nice One Beauty Digital Marketing Company.csv")
df.head()

,Date,Close,High,Low,Open,Volume
0,1/8/2025,45.500000,45.500000,38.500000,38.500000,1415516
1,1/9/2025,49.000000,54.400002,45.099998,50.000000,32034350
2,1/12/2025,50.599998,52.400002,49.000000,49.700001,8232695
3,1/13/2025,49.450001,51.299999,49.099998,50.799999,3191676
4,1/14/2025,54.299999,54.299999,49.599998,49.750000,5258995


In [30]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 267 entries, 0 to 266
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    267 non-null    str    
 1   Close   267 non-null    float64
 2   High    267 non-null    float64
 3   Low     267 non-null    float64
 4   Open    267 non-null    float64
 5   Volume  267 non-null    int64  
dtypes: float64(4), int64(1), str(1)
memory usage: 12.6 KB


**<mark style="background-color: #ffdf3d;color:black;border-radius:5px;opacity:1.0">2. Date 컬럼의 데이터타입을 str ➔ datetime  변경</mark>** 

**상태 : Date 컬럼이 str(문자열) 데이터 타입임을 확인.** :  주식데이터의 경우 시간의 흐름이 생명인 시계열(Time-Series) 데이터 <br> ➔ 문자열을 파이썬이 인식하는 `datetime 데이터타입`으로 바꿔야함.

1/14/2025 ➔ 월(m) / 일(d) / 연(Y)
- 전 :   Date    267 non-null    str 
- 후 : Date    267 non-null    datetime64[us]

In [31]:
# date 컬럼의 데이터타입을 str  datetime으로
df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%Y')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 267 entries, 0 to 266
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    267 non-null    datetime64[us]
 1   Close   267 non-null    float64       
 2   High    267 non-null    float64       
 3   Low     267 non-null    float64       
 4   Open    267 non-null    float64       
 5   Volume  267 non-null    int64         
dtypes: datetime64[us](1), float64(4), int64(1)
memory usage: 12.6 KB


**<mark style="background-color: #ffdf3d;color:black;border-radius:5px;opacity:1.0">3. Date 컬럼을 시간 순서대로 정렬</mark>** 

**reset_index(drop=True)**
데이터를 날짜순으로 새로 정렬했으니 원래 인덱스를 리셋해라

- sort_value('date')만 할 경우 ➔ 인덱스가 꼬임 & 조회 시 에러
    - 1번 줄: 2025-01-08
    - 2번 줄: 2025-01-09
    - 0번 줄: 2025-01-10
- `drop=True` : 기존 인덱스 그냥 버려라
    - 그냥 reset_index()만 실행하면 `기존의 인덱스를 index`라는 새로운 데이터 열로 만들어줌 

In [32]:
# 시계열 데이터 정렬하기 (과거에서 현재순)
df = df.sort_values('Date').reset_index(drop=True)
df

,Date,Close,High,Low,Open,Volume
0,2025-01-08,45.500000,45.500000,38.500000,38.500000,1415516
1,2025-01-09,49.000000,54.400002,45.099998,50.000000,32034350
2,2025-01-12,50.599998,52.400002,49.000000,49.700001,8232695
3,2025-01-13,49.450001,51.299999,49.099998,50.799999,3191676
4,2025-01-14,54.299999,54.299999,49.599998,49.750000,5258995
...,...,...,...,...,...,...
262,2026-01-25,18.260000,18.540001,18.150000,18.500000,806287
263,2026-01-26,18.250000,18.340000,18.100000,18.260000,308562
264,2026-01-27,18.260000,18.549999,17.299999,18.250000,1972097
265,2026-01-28,17.980000,18.299999,17.900000,18.260000,907635


<div style="color:white;display:fill;border-radius:8px;background-color:#323232;font-size:150%; letter-spacing:1.0px"><p style="padding: 12px;color:white;"><b><b><span style='color:white'><span style='color:#F1A424'>1.4 | </span></span></b>피쳐 엔지니어링_ 특성변수 만들기</b></p></div>

1. '일일 수익률(Daily Return)' 및 '로그 수익률(Log Return)'
2. 변동성 지표: 일일 수익률의 7일 rolling 표준편차
3. 변동성 지표: 일일 수익률의 30일 표준편차 <br> rolling window = 7 / 30
4. 시간 관련 메타데이터 추출 (연, 월, 월 이름, 요일, 분기) <br> .dt.

**<mark style="background-color: #ffdf3d;color:black;border-radius:5px;opacity:1.0"> 1. '일일 수익률(Daily Return)' 및 '로그 수익률(Log Return)'</mark>** 
- 판다스 내장함수 : `pct_change()`<br>
    - 한 객체 내에서 행과 행의 차이를 현재값과의 백분율로 출력하는 메서드
    - 계산 로직 : (다음행 - 현재행) / 현재행 
    - 첫 행의 경우 이전 행이 없으므로 `nan` 출력

- 로그 수익률 (Log Return)을 쓰는 이유 (로그 씌우는 이유):<br> 일반수익률의 문제 ➔ 50% 상승 다음날 50% 폭락일 경우 수익률이 0이 됨. <br> 하지만 수익률의 경우 50% 폭락 - >50만원, 50만원의 50% 폭등 75만원이 됨.
일반 수익률은 시간이 흐르면서 여러번 더하고 곱할 때 직관적인 계산이 틀어짐 
로그수익률의 경우 합산 결과가 정확히 0이 되는 대칭성을 가짐 

즉 요약 : 
1. 일일 수익률(Daily_Return): 사람이 눈으로 보고 "아, 오늘 주가가 몇 % 올랐구나/내렸구나" 직관적으로 이해하기 위한 변수.
2. 로그 수익률(Log_Return): 컴퓨터나 통계 모델이 "과거부터 현재까지의 전체 주가 흐름을 왜곡 없이 수학적으로 정확하게 연산"하기 위한 변수.

1. 일반 수식률 : 오늘 주가 - 어제 주가 / 어제 주가 
ex) 어제 주가 : 만원 오늘 주가 5000원
수익률 : 5000 - 10_000 / 10_000  - 50 


로그 수익률 = 


$$r_t = \ln\left(\frac{\text{오늘 종가}}{\text{어제 종가}}\right)$$

In [33]:
df['Daily_Return'] = df['Close'].pct_change() # 일일 수익률

판다스 내장 함수 : shift()
판다스에서 데이터를 아래로 미는 함수 ➔   `shift(1) ➔ 시계열 데이터가 날짜순으로 정렬되어 있기 때문에 오늘 줄 에서 데이터를 아래로 내리면 ➔ '어제 종가($P_{t-1}$)'를 가리키게 됨 . 

이걸 위해서 데이터를 최신 ➔ 오래된 순으로 정렬

오늘 종가($P_t$)를 방금 구한 어제 종가($P_{t-1}$)로 나눕니다 ($\frac{P_t}{P_{t-1}}$).

In [34]:
df['close_yesterday'] = df['Close'].shift(1) # 인덱스 0은 nan ➔ 이전 종가가 없기 때문

df['오늘_어제_종료가비율'] = df['Close']/df['close_yesterday']

df['Log_Return'] = np.log(df['Close']/df['Close'].shift(1)) # 로그 수익률

In [35]:
df.head()

,Date,Close,High,Low,Open,Volume,Daily_Return,close_yesterday,오늘_어제_종료가비율,Log_Return
0,2025-01-08,45.500000,45.500000,38.500000,38.500000,1415516,NaN,NaN,NaN,NaN
1,2025-01-09,49.000000,54.400002,45.099998,50.000000,32034350,0.076923,45.500000,1.076923,0.074108
2,2025-01-12,50.599998,52.400002,49.000000,49.700001,8232695,0.032653,49.000000,1.032653,0.032131
3,2025-01-13,49.450001,51.299999,49.099998,50.799999,3191676,-0.022727,50.599998,0.977273,-0.022989
4,2025-01-14,54.299999,54.299999,49.599998,49.750000,5258995,0.098079,49.450001,1.098079,0.093562


In [36]:
df.rename(columns={'Log_Retrun':'Log_Return'}, inplace=True)

In [37]:
df[['Date', 'Close', 'close_yesterday', '오늘_어제_종료가비율', 'Log_Return']].head()

,Date,Close,close_yesterday,오늘_어제_종료가비율,Log_Return
0,2025-01-08,45.500000,NaN,NaN,NaN
1,2025-01-09,49.000000,45.500000,1.076923,0.074108
2,2025-01-12,50.599998,49.000000,1.032653,0.032131
3,2025-01-13,49.450001,50.599998,0.977273,-0.022989
4,2025-01-14,54.299999,49.450001,1.098079,0.093562


**<mark style="background-color: #ffdf3d;color:black;border-radius:5px;opacity:1.0"> 2. 시계열 파생 변수 : 이동 변동성 : 7days, 30days </mark>** 

주식에서 변동성('Volatility') ➔ 주가가 얼마나 요동치는 지를 나타내는 위험도의 지표 


금융_주식 분석에서 중요한 개념 : rolling, window = 움직이는 틀

배경 : 주식 시장의 경우 하루 안에도 데이터(주가)가 변동이 심함 -> 당장 오늘 엄청 폭등했다고 해서 이게 진짜인지 일시적인 거품인지 알 수 없음.
➔ 오늘만 보지 말고 오늘을 포함한 최근 1주일 동안의 흐름을 묶어서 관찰하자 ➔ 이때 사용하는 판다스 도구 : rolling

rolling : 굴리다 미끄러지다 ➔ 고정된 자리에 가만히 있는 게 아니라 날짜가 흐름에 딸 ㅏ아래로 스르륵 미끄러지면서 움직이는 행위
window 창문의 크기 ➔ 한번에 내다볼 크기 (데이터 행의 개수) ex ) 7일치 ➔ window = 7

- 근데 7개 볼 게 없으면 nan 결측치 내뱉는다고 함~
- volatility : 변동성 : 금융에서 시간에 따른 일련의 거래 가격의 변동 정도

- 해석 : 
    - 주식시장에서 volatility(변동성) = risk(위험)
    - 변동성이 낮으면 ➔ 걔네 평균에서 떨어진 정도가 작다 = 표준편차가 작다 
    - 변동성이 크면 ➔ 표준편차가 크다. 가격의 오르락내리락이 심하다 

In [38]:
df['Rolling_Volatility_7days'] = df['Daily_Return'].rolling(window=7).std()

# 1. 데일리 리턴 ➔ 일일 수익률을 가지고 옴. 변화 비율을 비교하기 위해
# 2. rolling(window=7) ➔ [오늘+과거6일] 한 묶음으로 묶음
# 3. std() ➔ [오늘+과거6일] 즉 7일치 데이터의 표준 편차를 구함 ➔ 데이터가 평균에서 얼마나 멀리 떨어져서 흔들리는가. 
df['Rolling_Volatility_30days'] = df['Daily_Return'].rolling(window=30).std()

In [39]:
df.rename(columns={'Rolling_Volatility_7days':'Rolling_Volatility_7d','Rolling_Volatility_30days':'Rolling_Volatility_30d'}, inplace=True)

### <b><span style='color:#F1A424'> 데이터를 시간 단위로 쪼개기 : Extract temporal metadata</span></b>

시간 메타데이터(Temporal Metadata) :<br>
년/분기/월/주/일 등 시간적 패턴을 찾아내기 위해서 날짜(Date)컬럼을 여러개로 쪼개기<br>
`pd.to_datetime()` ➔ 날짜형 데이터 변환 ➔ 판다스에서 인식할 수 있는 datetime64 특수 타입이 됨.
- dt : Datetime 접근자
    - .strfime(), .day_name() 을 사용할 수 있음 .
    - dt.strftime('%b') : 날짜 형식을 원하는 문자열 모양으로 변환 str/ format / time : %b ➔ month를 축약형 영문으로 (Jan, Feb, Mar...)
    - dt.day_name() : 해당 날짜의 요일을 계산해서 풀 네임으로 영문 문자열을 변환 
    - dt.to_period('Q') : 날짜를 기반으로 몇분기(Quarter)에 속하는 지 계산 <br> (예: 2025-01-08이면 1분기니까 2025Q1로 변환)
    - .astype(str) : 문자열 타입으로 변환 

In [44]:
# 연도(Year)추출해서 새컬럼에 저장
df['Year'] = df['Date'].dt.year
# 날짜 데이터에서 월(Month) 정보만 추출해서 새컬럼에 저장
df['Month'] = df['Date'].dt.month

In [ ]:
df['Month_name'] = df['Date'].dt.strftime('%b') # 날짜를 원하는 문자열 포맷으로 변환 ex) 1월이면 Jan
df['Day_of_Week'] = df['Date'].dt.day_name() # 해당 날짜의 요일을 영문으로 반환  ex) Monday

In [42]:
# 분기 반환
df['Quarter'] = df['Date'].dt.to_period('Q').astype(str)

In [43]:
df.head()

,Date,Close,High,Low,Open,Volume,Daily_Return,close_yesterday,오늘_어제_종료가비율,Log_Return,Rolling_Volatility_7d,Rolling_Volatility_30d,Year,Month,Month_name,Day_of_Week,Quarter
0,2025-01-08,45.500000,45.500000,38.500000,38.500000,1415516,NaN,NaN,NaN,NaN,NaN,NaN,2025,1,Jan,Wednesday,2025Q1
1,2025-01-09,49.000000,54.400002,45.099998,50.000000,32034350,0.076923,45.500000,1.076923,0.074108,NaN,NaN,2025,1,Jan,Thursday,2025Q1
2,2025-01-12,50.599998,52.400002,49.000000,49.700001,8232695,0.032653,49.000000,1.032653,0.032131,NaN,NaN,2025,1,Jan,Sunday,2025Q1
3,2025-01-13,49.450001,51.299999,49.099998,50.799999,3191676,-0.022727,50.599998,0.977273,-0.022989,NaN,NaN,2025,1,Jan,Monday,2025Q1
4,2025-01-14,54.299999,54.299999,49.599998,49.750000,5258995,0.098079,49.450001,1.098079,0.093562,NaN,NaN,2025,1,Jan,Tuesday,2025Q1


### <b><span style='color:#F1A424'> Sanity Check: 논리적으로 말이 되는지 빠르게 확인: 기초검증절차 </span></b>

 데이터가 의도한 대로 잘 다듬어졌는지 최종적으로 확인 ➔ 새니티 체크  

In [ ]:
# df의 [전체 행수, 열(컬럼 수 )]
print("데이터 shape:", df.shape)

데이터 shape: (267, 17)


In [47]:
print("data Completeness check(=데이터의 완전성 체크_ missing value가 있는가):")
print(df.isnull().sum())

data Completeness check(=데이터의 완전성 체크_ missing value가 있는가):
Date                       0
Close                      0
High                       0
Low                        0
Open                       0
Volume                     0
Daily_Return               1
close_yesterday            1
오늘_어제_종료가비율                1
Log_Return                 1
Rolling_Volatility_7d      7
Rolling_Volatility_30d    30
Year                       0
Month                      0
Month_name                 0
Day_of_Week                0
Quarter                    0
dtype: int64


In [49]:
print("first 5 records")
df.head()

first 5 records


,Date,Close,High,Low,Open,Volume,Daily_Return,close_yesterday,오늘_어제_종료가비율,Log_Return,Rolling_Volatility_7d,Rolling_Volatility_30d,Year,Month,Month_name,Day_of_Week,Quarter
0,2025-01-08,45.500000,45.500000,38.500000,38.500000,1415516,NaN,NaN,NaN,NaN,NaN,NaN,2025,1,Jan,Wednesday,2025Q1
1,2025-01-09,49.000000,54.400002,45.099998,50.000000,32034350,0.076923,45.500000,1.076923,0.074108,NaN,NaN,2025,1,Jan,Thursday,2025Q1
2,2025-01-12,50.599998,52.400002,49.000000,49.700001,8232695,0.032653,49.000000,1.032653,0.032131,NaN,NaN,2025,1,Jan,Sunday,2025Q1
3,2025-01-13,49.450001,51.299999,49.099998,50.799999,3191676,-0.022727,50.599998,0.977273,-0.022989,NaN,NaN,2025,1,Jan,Monday,2025Q1
4,2025-01-14,54.299999,54.299999,49.599998,49.750000,5258995,0.098079,49.450001,1.098079,0.093562,NaN,NaN,2025,1,Jan,Tuesday,2025Q1


<div style="color:white;display:fill;border-radius:8px;background-color:#323232;font-size:150%; letter-spacing:1.0px"><p style="padding: 12px;color:white;"><b><b><span style='color:white'><span style='color:#F1A424'>1.5 | 
</span></span></b>  분석 함수 및 구조적 인사이트(Analytical Functions & Structural Insights)</b></p></div>

분석의 4가지 관점
1. 구성 분석 (Composition Analysis) : 전체적인 구조, 데이터 타입, 총 주식 거래량 평가
2. 분포 분석 (Distribution Analysis) : 가격, 걸량, 수익률의 형태(왜도, 첨도 및 확률 밀도)
3. 비교 분석 (Comparison Analysis) : 월별 분기별 주가 성과와 변동성을 벤치마킹(비교)
4. 관계 분석 (Relationship Analysis) : 거래량, 주가 움직임, 시장 변동성 간의 상관관계 

In [52]:
#1.5.1 구성 분석 : 데이터프레임의 구성을 분석, 전체 집계 지표를 다룸

# 분석 함수 정의
def analyze_composition(data):
    print("==데이터셋 구성 요약 ==")
    print(f"총 주식 거래 일수(Total trading sessions_observations): {len(data)}")
    print(f"전체 컬럼 수 (피쳐 수) : {data.shape[1]}")
    print(f"피쳐 종류 : {list(data.columns)}")
    print("\nFeature type & 메모리 확인") 
    print(data.info())# 각 컬럼의 데이터타입, 그리고 메모리를 얼마나 잡아먹나
    
    
    total_volume_traded = data['Volume'].sum()  # 누적 주식 거래량(Volume) 
    print(f"\n누적 주식 거래량(2025-2026): {total_volume_traded:,} shares") # :, ➔ 3자리마다 콤마
    print(f"최초 공모가(IPO 제안 금액) : SAR 35.00")
    print(f"최종 마감 주가(종가): SAR {data['Close'].iloc[-1]:.2f}") # 종가의 뒤에서 첫번째 즉 가장 최신의 종료가를 가져오되 소수점 2자리까지만 보여주삼
    
    
analyze_composition(df)

==데이터셋 구성 요약 ==
총 주식 거래 일수(Total trading sessions_observations): 267
전체 컬럼 수 (피쳐 수) : 17
피쳐 종류 : ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Daily_Return', 'close_yesterday', '오늘_어제_종료가비율', 'Log_Return', 'Rolling_Volatility_7d', 'Rolling_Volatility_30d', 'Year', 'Month', 'Month_name', 'Day_of_Week', 'Quarter']

Feature type & 메모리 확인
<class 'pandas.DataFrame'>
RangeIndex: 267 entries, 0 to 266
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Date                    267 non-null    datetime64[us]
 1   Close                   267 non-null    float64       
 2   High                    267 non-null    float64       
 3   Low                     267 non-null    float64       
 4   Open                    267 non-null    float64       
 5   Volume                  267 non-null    int64         
 6   Daily_Return            266 non-null    float64       
 7   close_yesterday   

## 2. 분포 분석 (Distribution Analysis) : 가격, 걸량, 수익률의 형태(왜도, 첨도 및 확률 밀도)

가격, 일일수익률(Daily Return)거래량(Volume)의 분포를 분석 ➔ 통계적 지표들(평균, 표준편차, 왜도, 첨도)을 계산하고 확률 밀도 그래프(Probability Density Plots)를 생성

왜도(Skewness) 및 첨도(Kurtosis) 계산:
왜도 : 데이터 분포 좌우 비대칭 정도를 표현하는 척도 ➔ 데이터의 분포가 얼마나 대칭인지 아닌지, 정규분포(좌우대칭)일 수록 왜도값은 작아짐. 한쪽으로 심하게 몰려있으면 왜도값 증가
- 양의 왜도 > 0 : 
    1. 오른쪽으로 긴 꼬리
    2. 평균 > 중앙값 
- 왜도 = 0 :
    1. 좌우 대칭
    2. 평균 = 중앙값 
- 음의 왜도 <0> : 음수 & 좌측 편향 
    1. 왼쪽으로 긴 꼬리
    2. 평균 < 중앙값 < 최빈값  
<img src = "https://dthumb-phinf.pstatic.net/?src=%22http%3A%2F%2Fwww.statisticshowto.com%2Fwp-content%2Fuploads%2F2014%2F02%2Fpearson-mode-skewness.jpg%22&type=m10000_10000">

첨도 : 분포가 정규분포보다 얼마나 뾰족 or 완만한 정도를 나타내는 척도 즉 얼마나 중심에 집중적으로 몰려있는가 
데이터가 중심에 많이 몰려있을 수록 뾰족 = 양의 첨도 !, 두루 퍼지면 음의 첨도 

In [ ]:
#  분포 분석

def analyze_distribution(data):
    
    # 1. 기본적인 데이터 기술통계량 print
    stats_df = data[['Open','High','Low','Close', 'Volume', 'Daily_Return']].describe()
    print("=== FINANCIAL STATISTICAL SUMMARY === ")
    print("=== 금융 통계 요약 SUMMARY === ")
    print(stats_df)
    
    # 2. 왜도와 첨도 계산
    print(f"\n종가 데이터 왜도(Skewness): {data['Close'].skew():.4f}")
    print(f"일일 수익률 왜도(Skewness): {data['Daily_Return'].dropna().skew():.4f}")
    print(f"일일 수익률 첨도(Kurtosis) : {data['Daily_Return'].dropna().kurtosis():.4f}")
    
    # 3. 보고서용 서브 플롯(1행 3열) 틀 짜기
    fig, axes = plt.subplot(1,3, figsize=(18,5)) # 왼쪽부터 차례대로 axes[0]-[2]
    
    # 0번방 : axes[0] : 종가 확률 밀도
    # 히스토그램(막대그래프)를 그리되, 데이터의 흐름을 부드럽게 하는 kde=TRUE(확률밀도함수 선을 함께 얹어라.
    # 세로축을 단순 개수가 아니라 확률 비중으로 (STAT="density")
    sns.histplot(data['Close'], kde=True, ax=axes[0], color='#2c3e50', stat= "density")    
    axes[0].set_title("종가 확률 밀도 분포", fontsize=12, fontweight='bold')
    
    